# Vector store comparison

Walkthrough for the offline resume RAG pipeline. It imports tested helpers from `src` and compares character vs recursive splitting, then Chroma vs FAISS vs optional Pinecone.

The notebook does not embed API keys. Pinecone and OpenAI run only when the matching environment variable is set; otherwise those steps skip or use the extractive fallback.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "src" / "config.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import os
os.chdir(root)
print(f"project_root={root}")


## Section 1 — Setup and ingestion

Load `Settings` from the environment (no hardcoded key) and read only the PDFs in `data/resumes/`. The out-of-domain text file stays in `data/ood/`.


In [ ]:
from src.config import Settings
from src.ingest import load_pdf_resumes, split_documents
from src.retrieve import (
    build_faiss_index,
    get_chroma_count,
    get_embeddings,
    init_chroma,
    query_store,
    upsert_and_query_pinecone,
    upsert_chroma,
)
from src.generate import generate_answer

settings = Settings()
print(f"has_pinecone={settings.has_pinecone} has_openai={settings.has_openai}")
print(f"embedding_model={settings.embedding_model_name} dim={settings.embedding_dim}")

docs = load_pdf_resumes(settings.resumes_dir)
candidates = sorted({doc.metadata["candidate_id"] for doc in docs})
print(f"loaded_pages={len(docs)} candidates={len(candidates)}")
print(candidates)
assert docs, "expected PDF pages"
assert all(str(doc.metadata["source_file"]).endswith(".pdf") for doc in docs)
assert (settings.ood_dir / "state_of_the_union.txt").is_file()
assert not any("state_of_the_union" in str(doc.metadata.get("source_file", "")) for doc in docs)


## Section 2 — Splitter comparison

Split the same pages with `character` and `recursive` (`chunk_size=1000`, `chunk_overlap=100`). `char_tokens` and `recursive_tokens` are the word-count lists for those chunks.


In [ ]:
char_chunks, char_ids, char_tokens = split_documents(
    docs,
    method="character",
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
)
recursive_chunks, recursive_ids, recursive_tokens = split_documents(
    docs,
    method="recursive",
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
)

def _stats(name, tokens):
    if not tokens:
        print(f"{name}: no chunks")
        return
    average = sum(tokens) / len(tokens)
    print(
        f"{name}: chunks={len(tokens)} min_words={min(tokens)} "
        f"max_words={max(tokens)} mean_words={average:.1f}"
    )

_stats("character", char_tokens)
_stats("recursive", recursive_tokens)
assert len(char_tokens) == len(char_chunks) == len(char_ids)
assert len(recursive_tokens) == len(recursive_chunks) == len(recursive_ids)
assert len(recursive_chunks) > 1
assert all(chunk.page_content.startswith("[Candidate: ") for chunk in recursive_chunks)
assert all(chunk.page_content.startswith("[Candidate: ") for chunk in char_chunks)


## Section 3 — Idempotent Chroma indexing

Index the recursive chunks twice with deterministic ids. The collection count after the second upsert stays equal to the chunk count.


In [ ]:
embeddings = get_embeddings(settings)
chroma_db = init_chroma(settings, embeddings)
try:
    chroma_db.delete_collection()
except Exception as exc:
    print(f"fresh collection: {exc}")
chroma_db = init_chroma(settings, embeddings)

count_first = upsert_chroma(chroma_db, recursive_chunks, recursive_ids)
count_second = upsert_chroma(chroma_db, recursive_chunks, recursive_ids)
print(f"chroma_count_after_first={count_first}")
print(f"chroma_count_after_second={count_second}")
print(f"get_chroma_count={get_chroma_count(chroma_db)}")
assert count_first == count_second == len(recursive_chunks)

query = "Python engineer with AWS and machine learning experience"
chroma_results = query_store(chroma_db, query, k=3)
print(f"chroma_hits={len(chroma_results)}")
for rank, (doc, score) in enumerate(chroma_results, start=1):
    print(rank, doc.metadata.get("candidate_id"), round(float(score), 4))


## Section 4 — FAISS and optional Pinecone

Build a FAISS index, print `faiss_results` from a fresh query, and skip Pinecone with a note when `PINECONE_API_KEY` is unset. The table compares the top hits and query latency.


In [ ]:
import time

faiss_db = build_faiss_index(recursive_chunks, embeddings, recursive_ids)

def _timed(label, fn):
    started = time.perf_counter()
    rows = fn()
    elapsed_ms = (time.perf_counter() - started) * 1000
    return rows, elapsed_ms

chroma_results, chroma_ms = _timed("chroma", lambda: query_store(chroma_db, query, k=3))
faiss_results, faiss_ms = _timed("faiss", lambda: query_store(faiss_db, query, k=3))
print("faiss_results")
print(faiss_results)

if settings.has_pinecone:
    pinecone_started = time.perf_counter()
    pinecone_matches = upsert_and_query_pinecone(
        recursive_chunks,
        recursive_ids,
        embeddings,
        query,
        settings,
        k=3,
    )
    pinecone_ms = (time.perf_counter() - pinecone_started) * 1000
else:
    pinecone_matches = []
    pinecone_ms = None
    print("Pinecone skipped: PINECONE_API_KEY is not set.")

header = f"{'store':<10} {'rank':<4} {'candidate':<22} {'score':<10} latency_ms"
print()
print(header)

def _hit_line(store, rank, candidate, score, latency):
    score_text = "" if score is None else f"{float(score):.4f}"
    latency_text = "" if latency is None else f"{latency:.1f}"
    return f"{store:<10} {rank:<4} {str(candidate):<22} {score_text:<10} {latency_text}"

for rank, (doc, score) in enumerate(chroma_results, start=1):
    latency = chroma_ms if rank == 1 else None
    print(_hit_line("chroma", rank, doc.metadata.get("candidate_id"), score, latency))
for rank, (doc, score) in enumerate(faiss_results, start=1):
    latency = faiss_ms if rank == 1 else None
    print(_hit_line("faiss", rank, doc.metadata.get("candidate_id"), score, latency))
if pinecone_matches:
    for rank, match in enumerate(pinecone_matches, start=1):
        metadata = match.get("metadata") or {}
        latency = pinecone_ms if rank == 1 else None
        candidate = metadata.get("candidate_id", match.get("id"))
        print(_hit_line("pinecone", rank, candidate, match.get("score"), latency))
else:
    print(_hit_line("pinecone", "-", "skipped", None, None))


## Section 5 — End-to-end RAG answer

Pass the top Chroma chunks into `generate_answer`. With no OpenAI key this prints the extractive `RAGAnswer`. With a key, it prints the structured model returned by the API.


In [ ]:
top_docs = [doc for doc, _score in chroma_results]
answer = generate_answer(query, top_docs, settings)
print(answer.model_dump_json(indent=2))
assert answer.query == query
assert answer.summary
if not settings.has_openai:
    assert len(answer.matches) == len(top_docs)
